In [ ]:
#cutoff

import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, confusion_matrix

# ================= CONFIG =================
LABEL_CSV = "../LiverFibrosisCohort/LiverFibrosisCohort_MasterTable_Proccessed_527.csv"
RAD_ROOT = "/mnt/12T_Ironwolf/LiverData/Radiomics"
OUTPUT_CSV = "../Results/rad_mlp_results_with_sen_spe.csv"

MODALITIES = ["ADC", "T1", "T2"]
DEV_SITES = ["CCHMC", "WISC","MICH"]

ID_COLUMN = "Image ID"
LABEL_COLUMN = "Fibrosis_Label"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EPOCHS = 50
LR = 1e-3
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

# ================= ID =================
def normalize_id(pid):
    pid = str(pid)

    m = re.match(r"PT-(\d{3}-\d{4})-", pid)
    if m:
        return m.group(1)

    m = re.match(r"PT-(M\d+)-", pid)
    if m:
        return m.group(1)

    return pid


def get_site(pid):
    pid = str(pid)

    if re.match(r"^\d{3}-\d{4}$", pid):
        return "CCHMC"
    elif pid.startswith("GJ"):
        return "WISC"
    elif pid.startswith("C_"):
        return "NYU"
    elif re.match(r"^\d+$", pid) or pid.startswith("M"):
        return "MICH"
    else:
        return "UNKNOWN"


# ================= LABEL =================
def binarize_fibrosis(x):
    if pd.isna(x):
        return np.nan

    x = str(x)
    m = re.search(r"\d+", x)
    if not m:
        return np.nan

    val = int(m.group())

    if val in [0, 1]:
        return 0
    elif val in [2, 3, 4]:
        return 1
    return np.nan


# ================= LOAD LABELS =================
labels_df = pd.read_csv(LABEL_CSV)
labels_df = labels_df.rename(columns={ID_COLUMN: "id"})

labels_df["id"] = labels_df["id"].astype(str).apply(normalize_id)
labels_df["label"] = labels_df[LABEL_COLUMN].apply(binarize_fibrosis)

labels_df = labels_df.dropna(subset=["label"])
labels_df["label"] = labels_df["label"].astype(int)
labels_df["site"] = labels_df["id"].apply(get_site)

labels_df = labels_df[["id", "label", "site"]]

# DEV cohort
labels_df = labels_df[labels_df["site"].isin(DEV_SITES)].copy()

print(f"DEV size: {len(labels_df)}")
print(labels_df["label"].value_counts())


# ================= MODEL =================
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()

        if hidden_dim == 1:
            self.net = nn.Linear(in_dim, 1)
        else:
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, 1)
            )

    def forward(self, x):
        return self.net(x).squeeze(1)


# ================= TRAIN =================
def train_eval(X, y, hidden_dim):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    all_probs = np.zeros(len(y), dtype=float)

    for tr, val in skf.split(X, y):
        X_tr, X_val = X[tr], X[val]
        y_tr, y_val = y[tr], y[val]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_val = scaler.transform(X_val)

        X_tr = torch.tensor(X_tr, dtype=torch.float32).to(DEVICE)
        y_tr = torch.tensor(y_tr, dtype=torch.float32).to(DEVICE)
        X_val = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)

        model = MLP(X.shape[1], hidden_dim).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        loss_fn = nn.BCEWithLogitsLoss()

        for _ in range(EPOCHS):
            model.train()
            logits = model(X_tr)
            loss = loss_fn(logits, y_tr)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            probs = torch.sigmoid(model(X_val)).cpu().numpy()

        all_probs[val] = probs

    preds = (all_probs > 0.5).astype(int)

    bal_acc = balanced_accuracy_score(y, preds)
    auc = roc_auc_score(y, all_probs)

    # confusion matrix: [[TN, FP], [FN, TP]]
    tn, fp, fn, tp = confusion_matrix(y, preds, labels=[0, 1]).ravel()

    sen = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spe = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    return bal_acc, auc, sen, spe


# ================= RUN =================
HIDDEN_SIZES = [32, 16, 8, 1]

results = {m: {"auc": [], "bal_acc": [], "sen": [], "spe": []} for m in MODALITIES}
rows = []

for modality in MODALITIES:
    print(f"\n===== RADIOMICS {modality} =====")

    liver_path = os.path.join(RAD_ROOT, f"{modality}_liver_features.csv")
    spleen_path = os.path.join(RAD_ROOT, f"{modality}_spleen_features.csv")

    liver_df = pd.read_csv(liver_path)
    spleen_df = pd.read_csv(spleen_path)

    # normalize IDs
    liver_df["ID"] = liver_df["ID"].astype(str).apply(normalize_id)
    spleen_df["ID"] = spleen_df["ID"].astype(str).apply(normalize_id)

    liver_df = liver_df.rename(columns={"ID": "id"})
    spleen_df = spleen_df.rename(columns={"ID": "id"})

    # merge organs
    feat_df = liver_df.merge(spleen_df, on="id", suffixes=("_liver", "_spleen"))

    # merge labels
    merged = labels_df.merge(feat_df, on="id", how="inner")

    print(f"Matched: {len(merged)}")

    if len(merged) == 0:
        print(f"⚠️ Skipping {modality}")
        continue

    feature_cols = [c for c in merged.columns if c not in ["id", "label", "site"]]

    X = merged[feature_cols].values
    y = merged["label"].values

    print(f"{modality} radiomics shape: {X.shape}")

    # handle NaNs
    X = np.nan_to_num(X)

    for h in HIDDEN_SIZES:
        bal_acc, auc, sen, spe = train_eval(X, y, h)

        print(
            f"MLP ({X.shape[1]} → {h} → 1): "
            f"Balanced Acc = {bal_acc:.4f}, "
            f"AUC = {auc:.4f}, "
            f"Sensitivity = {sen:.4f}, "
            f"Specificity = {spe:.4f}"
        )

        results[modality]["auc"].append(auc)
        results[modality]["bal_acc"].append(bal_acc)
        results[modality]["sen"].append(sen)
        results[modality]["spe"].append(spe)

        rows.append({
            "modality": modality,
            "n_features": X.shape[1],
            "hidden_dim": h,
            "balanced_accuracy": bal_acc,
            "auc": auc,
            "sensitivity": sen,
            "specificity": spe
        })

# ================= SAVE CSV =================
results_df = pd.DataFrame(rows)
results_df.to_csv(OUTPUT_CSV, index=False)

print(f"\nSaved results to: {OUTPUT_CSV}")
print(results_df)

DEV size: 481
label
1    259
0    222
Name: count, dtype: int64

===== RADIOMICS ADC =====
Matched: 149
ADC radiomics shape: (149, 200)
MLP (200 → 32 → 1): Balanced Acc = 0.5683, AUC = 0.5314, Sensitivity = 0.6154, Specificity = 0.5211
MLP (200 → 16 → 1): Balanced Acc = 0.5612, AUC = 0.5457, Sensitivity = 0.6154, Specificity = 0.5070
MLP (200 → 8 → 1): Balanced Acc = 0.5382, AUC = 0.5630, Sensitivity = 0.6538, Specificity = 0.4225
MLP (200 → 1 → 1): Balanced Acc = 0.5401, AUC = 0.5506, Sensitivity = 0.6154, Specificity = 0.4648

===== RADIOMICS T1 =====
Matched: 310
T1 radiomics shape: (310, 200)
MLP (200 → 32 → 1): Balanced Acc = 0.5252, AUC = 0.5393, Sensitivity = 0.6391, Specificity = 0.4113
MLP (200 → 16 → 1): Balanced Acc = 0.5471, AUC = 0.5689, Sensitivity = 0.6686, Specificity = 0.4255
MLP (200 → 8 → 1): Balanced Acc = 0.5435, AUC = 0.5600, Sensitivity = 0.6686, Specificity = 0.4184
MLP (200 → 1 → 1): Balanced Acc = 0.5269, AUC = 0.5597, Sensitivity = 0.5858, Specificity = 0.468

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt

# create output folder
PLOT_DIR = "../Results/rad_plots"
os.makedirs(PLOT_DIR, exist_ok=True)

BAR_WIDTH = 0.35
TITLE_FONT = 26
LABEL_FONT = 20
TICK_FONT = 18
VALUE_FONT = 16

MODALITY_ORDER = ["T1", "T2", "ADC"]
PANEL_LABELS = ["(a)", "(b)", "(c)"]


def modality_display_name(modality):
    modality = str(modality).upper()
    if modality == "T1":
        return "T1w"
    elif modality == "T2":
        return "T2w"
    elif modality == "ADC":
        return "DWI"
    return modality


def plot_combined_metric(results, hidden_sizes, metric_key, y_label, out_path):
    labels = [str(h) for h in hidden_sizes]
    x = np.arange(len(labels))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharey=True)

    # collect all valid values first so the y-limit is shared across subplots
    all_vals = []
    for modality in MODALITY_ORDER:
        if modality in results and metric_key in results[modality]:
            vals = results[modality][metric_key]
            if len(vals) > 0:
                all_vals.extend(vals)

    if len(all_vals) == 0:
        print(f"⚠️ No valid data found for {metric_key}, skipping plot.")
        plt.close()
        return

    ymax = max(0.75, float(np.nanmax(all_vals)) + 0.08)
    ymax = min(1.0, ymax)

    for i, (ax, modality, panel) in enumerate(zip(axes, MODALITY_ORDER, PANEL_LABELS)):
        ax.set_title(
            f"{panel} {modality_display_name(modality)} (Radiomic Features)",
            fontsize=TITLE_FONT,
            pad=14
        )
        ax.set_xlabel("Hidden Layer Size", fontsize=LABEL_FONT)
        ax.set_ylim(0, ymax)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=TICK_FONT)

        if i == 0:
            ax.set_ylabel(y_label, fontsize=LABEL_FONT)
            ax.tick_params(axis="y", labelsize=TICK_FONT)
        else:
            ax.tick_params(axis="y", left=False, labelleft=False)

        if modality not in results or metric_key not in results[modality]:
            ax.text(
                0.5, 0.5, "No data",
                ha="center", va="center",
                transform=ax.transAxes, fontsize=18
            )
            continue

        vals = results[modality][metric_key]

        if len(vals) == 0:
            ax.text(
                0.5, 0.5, "No data",
                ha="center", va="center",
                transform=ax.transAxes, fontsize=18
            )
            continue

        bars = ax.bar(x, vals, width=BAR_WIDTH)

        for bar, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    v + 0.008,
                    f"{v:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=VALUE_FONT,
                )

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


# ===== save combined AUROC figure =====
plot_combined_metric(
    results=results,
    hidden_sizes=HIDDEN_SIZES,
    metric_key="auc",
    y_label="AUROC",
    out_path=os.path.join(PLOT_DIR, "combined_auc_radiomic_features.png"),
)

# ===== save combined Balanced Accuracy figure =====
plot_combined_metric(
    results=results,
    hidden_sizes=HIDDEN_SIZES,
    metric_key="bal_acc",
    y_label="Balanced Accuracy",
    out_path=os.path.join(PLOT_DIR, "combined_bal_acc_radiomic_features.png"),
)